# S01E05 — Railway

**Zadanie:** Aktywować trasę kolejową **X-01** za pomocą samo-dokumentującego się API.  
API regularnie zwraca 503 (symulacja przeciążenia) i ma restrykcyjne limity zapytań.

**Strategia:**
1. Zacznij od akcji `help` — API opisuje sam siebie
2. Podążaj krokami z dokumentacji
3. Obsługuj 503 z retry + exponential backoff
4. Sprawdzaj nagłówki rate-limit po każdym żądaniu
5. Agent LLM interpretuje odpowiedzi i decyduje co dalej

In [ ]:
import os, json, time, requests
from anthropic import Anthropic
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

API_KEY    = os.getenv("AI_DEVS_API_KEY")
client     = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
VERIFY_URL = "https://hub.ag3nts.org/verify"
TASK       = "railway"

print("Konfiguracja OK.")

In [ ]:
def call_railway(answer: dict, max_retries: int = 8) -> tuple[dict, dict]:
    """
    Wysyła żądanie do API railway z obsługą 503 i rate limitów.
    Zwraca (body_json, headers).
    """
    payload  = {"apikey": API_KEY, "task": TASK, "answer": answer}
    backoff  = 2

    for attempt in range(max_retries):
        resp = requests.post(VERIFY_URL, json=payload)
        headers = dict(resp.headers)

        if resp.status_code == 503:
            print(f"  503 — retry za {backoff}s (próba {attempt+1}/{max_retries})")
            time.sleep(backoff)
            backoff = min(backoff * 2, 30)
            continue

        if resp.status_code == 429:
            # Odczytaj reset time z nagłówków
            reset_after = int(headers.get("X-RateLimit-Reset-After",
                             headers.get("Retry-After", "10")))
            print(f"  429 — rate limit, czekam {reset_after}s")
            time.sleep(reset_after)
            continue

        try:
            body = resp.json()
        except Exception:
            body = {"raw": resp.text}

        # Log rate-limit headers jeśli obecne
        rl_remaining = headers.get("X-RateLimit-Remaining")
        rl_reset     = headers.get("X-RateLimit-Reset-After")
        if rl_remaining:
            print(f"  [rate limit] remaining={rl_remaining}, reset_after={rl_reset}s")

        return body, headers

    raise RuntimeError(f"Nie udało się po {max_retries} próbach")


print("Funkcja call_railway gotowa.")

In [ ]:
# Krok 1: pobierz dokumentację API
print("Wywołuję akcję help...")
help_resp, _ = call_railway({"action": "help"})
print("Odpowiedź help:")
print(json.dumps(help_resp, ensure_ascii=False, indent=2))

In [ ]:
# Krok 2: Agent LLM interpretuje dokumentację i wykonuje kolejne kroki

SYSTEM = """Jesteś agentem do aktywacji trasy kolejowej X-01.
Otrzymujesz odpowiedzi z API i decydujesz co wywołać dalej.
Każda Twoja odpowiedź to WYŁĄCZNIE JSON z polem 'action' i opcjonalnymi parametrami,
zgodnie z dokumentacją API. Gdy zobaczysz flagę {FLG:...} w odpowiedzi, zwróć {"done": true}.
Jeśli coś jest niejasne, przeczytaj uważnie komunikat błędu — zazwyczaj wskazuje co poprawić."""

messages = [
    {"role": "user", "content": f"Dokumentacja API railway:\n{json.dumps(help_resp, ensure_ascii=False)}\n\nTwój cel: aktywować trasę X-01. Zacznij od pierwszego kroku."}
]

MAX_STEPS = 30
history   = []

for step in range(MAX_STEPS):
    # Zapytaj agenta co zrobić dalej
    ai_resp = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=512,
        system=SYSTEM,
        messages=messages
    )
    agent_text = ai_resp.content[0].text.strip()
    print(f"\nKrok {step+1} — Agent: {agent_text}")

    # Parsuj JSON z odpowiedzi agenta
    try:
        start  = agent_text.find("{")
        end    = agent_text.rfind("}") + 1
        action = json.loads(agent_text[start:end])
    except Exception as e:
        print(f"  Błąd parsowania JSON agenta: {e}")
        break

    if action.get("done"):
        print("\nAgent zgłosił zakończenie!")
        break

    # Wywołaj API
    api_resp, _ = call_railway(action)
    api_text    = json.dumps(api_resp, ensure_ascii=False)
    history.append({"step": step+1, "action": action, "result": api_resp})
    print(f"  API: {api_text[:300]}")

    # Sprawdź czy jest flaga
    if "FLG:" in api_text:
        import re
        flags = re.findall(r'\{FLG:[^}]+\}', api_text)
        print(f"\nFLAGA: {flags}")
        break

    # Przekaż wynik do agenta
    messages.append({"role": "assistant", "content": agent_text})
    messages.append({"role": "user",      "content": f"Odpowiedź API:\n{api_text}\n\nCo robisz dalej?"})

print("\nHistoria kroków:", len(history))